# 🧪 Brouillon 03 — Transfer Learning ResNet

## Objectif

Comparer **ResNet18** (ImageNet) à notre baseline CNN pour l'extraction de features sur visages alignés.

In [ ]:
from pathlib import Path
import torch
import torch.nn as nn
from torchvision import models, transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name in ('drafts', 'notebooks'):
    PROJECT_ROOT = PROJECT_ROOT.parents[1] if PROJECT_ROOT.name == 'drafts' else PROJECT_ROOT.parent
DATASET_DIR = PROJECT_ROOT / 'dataset'

device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')

transform = transforms.Compose([
    transforms.Resize((160, 160)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

if DATASET_DIR.exists():
    ds = ImageFolder(str(DATASET_DIR), transform=transform)
    loader = DataLoader(ds, batch_size=8, shuffle=True)
    resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    resnet.fc = nn.Linear(resnet.fc.in_features, len(ds.classes))
    resnet = resnet.to(device)
    print('ResNet18 adapté — classes:', len(ds.classes))
else:
    print('dataset/ non trouvé')

In [ ]:
# Comparaison qualitative (brouillon)
comparison = {
    'Modèle': ['CNN baseline', 'ResNet18 TL', 'FaceNet (cible)'],
    'Pré-entraînement': ['Aucun', 'ImageNet', 'VGGFace2 (visages)'],
    'Embedding': ['Logits classe', 'Logits classe', 'Vecteur 512-D'],
    'Adapté biométrie': ['Faible', 'Moyen', 'Élevé'],
}
import pandas as pd
pd.DataFrame(comparison)

## Conclusion

ResNet améliore nettement le baseline, mais reste **orienté classification ImageNet**, pas métrique identité.

➡️ Choix final : **InceptionResnetV1 (facenet-pytorch)** pour des embeddings normalisés et comparaison par distance L2.